# Traffic Forecasting Model Training

Notebook-first training workflow for the Cognitive Traffic Analytics Platform. It is designed for Kaggle/Colab experimentation while reusing the same Python modules used by the local CLI and Airflow.

For Kaggle: upload a Dataset containing Gold files such as `train_features_15m.parquet` or `train_features_15m.csv`, then update `KAGGLE_DATASET_NAME` in the setup cell if your dataset slug is different.

## 1. Project setup

In [ ]:
from pathlib import Path
import json
import os
import sys

KAGGLE_MODE = Path('/kaggle/input').exists()
KAGGLE_DATASET_NAME = 'traffic-gold-dataset'

if KAGGLE_MODE:
    DATA_DIR = Path('/kaggle/input') / KAGGLE_DATASET_NAME
    OUTPUT_DIR = Path('/kaggle/working')
    METADATA_DIR = OUTPUT_DIR
else:
    DATA_DIR = Path('../data/gold')
    OUTPUT_DIR = Path('../models/artifacts')
    METADATA_DIR = Path('../models/metadata')

# Make local execution robust whether the notebook is launched from repo root or notebooks/.
if not KAGGLE_MODE and not DATA_DIR.exists() and Path('data/gold').exists():
    DATA_DIR = Path('data/gold')
    OUTPUT_DIR = Path('models/artifacts')
    METADATA_DIR = Path('models/metadata')

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'ml').exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print({'kaggle_mode': KAGGLE_MODE, 'data_dir': str(DATA_DIR), 'output_dir': str(OUTPUT_DIR), 'metadata_dir': str(METADATA_DIR)})

## 2. Load dataset

The notebook prefers `train_features_15m.parquet` and falls back to CSV. On Kaggle, place that file in the uploaded Dataset.

In [ ]:
from ml.training.train_utils import load_training_dataset, train_baseline_model, train_lightgbm_model
from ml.training.features import infer_target_column, select_feature_columns, time_ordered_split
from ml.training.metrics import evaluate_regression
from ml.training.model_io import build_model_manifest, save_feature_importance, save_metrics, save_model_artifact, save_model_manifest
from ml.tracking.mlflow_utils import log_training_run

dataset_candidates = [DATA_DIR / 'train_features_15m.parquet', DATA_DIR / 'train_features_15m.csv']
dataset_path = next((path for path in dataset_candidates if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(f'No training dataset found in {DATA_DIR}. Expected train_features_15m.parquet or train_features_15m.csv')

df = load_training_dataset(dataset_path)
print(f'Loaded {len(df):,} rows and {len(df.columns):,} columns from {dataset_path}')

## 3. Inspect data

In [ ]:
display(df.head())
print('Dataset shape:', df.shape)
missing = df.isna().mean().sort_values(ascending=False).head(20)
display(missing.to_frame('missing_ratio'))

## 4. Feature selection

Target and feature columns are inferred by shared production modules to keep notebook and CLI behavior aligned.

In [ ]:
target_column = infer_target_column(df, preferred='target_speed')
selection = select_feature_columns(df.dropna(subset=[target_column]).copy(), target_column)
print('Target:', target_column)
print('Feature count:', len(selection.feature_columns))
print('Numeric features:', len(selection.numeric_columns))
print('Categorical features:', len(selection.categorical_columns))
selection.feature_columns[:20]

## 5. Train/validation split

A time-ordered split is used when timestamp columns are available.

In [ ]:
work_df = df.dropna(subset=[target_column]).copy()
train_df, validation_df = time_ordered_split(work_df, validation_fraction=0.2)
print({'train_rows': len(train_df), 'validation_rows': len(validation_df)})

## 6. Baseline model

The baseline predicts the median target speed. This gives a simple reference before LightGBM.

In [ ]:
baseline_model, baseline_metrics = train_baseline_model(train_df, validation_df, selection)
baseline_metrics

## 7. LightGBM model training

In [ ]:
result = train_lightgbm_model(work_df, target_column=target_column, validation_fraction=0.2, random_state=42)
print({'train_rows': result.train_rows, 'validation_rows': result.validation_rows, 'feature_count': len(result.feature_selection.feature_columns)})

## 8. Evaluation metrics

In [ ]:
print('Baseline metrics')
display(baseline_metrics)
print('LightGBM metrics')
display(result.metrics)

## 9. Feature importance

In [ ]:
top_importance = result.feature_importance.head(25)
display(top_importance)
try:
    ax = top_importance.sort_values('importance').plot.barh(x='feature', y='importance', figsize=(10, 7), legend=False, title='Top LightGBM features')
    fig = ax.get_figure()
    fig.tight_layout()
    feature_importance_png = OUTPUT_DIR / 'feature_importance.png'
    fig.savefig(feature_importance_png, dpi=140)
    print('Saved plot:', feature_importance_png)
except Exception as exc:
    feature_importance_png = None
    print('Plot skipped:', exc)

## 10. Error analysis

In [ ]:
display(result.predictions.sort_values('absolute_error', ascending=False).head(20))
if 'city' in result.predictions.columns:
    display(result.predictions.groupby('city')['absolute_error'].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False))

## 11. Save model artifact

On Kaggle, outputs are written to `/kaggle/working`. Locally, the model goes to `models/artifacts`.

In [ ]:
model_path = save_model_artifact(result.model, OUTPUT_DIR, 'traffic_model.joblib')
metrics_path = save_metrics(result.metrics, METADATA_DIR if not KAGGLE_MODE else OUTPUT_DIR, 'metrics.json')
importance_path = save_feature_importance(result.feature_importance, METADATA_DIR if not KAGGLE_MODE else OUTPUT_DIR, 'feature_importance.csv')
print({'model_path': str(model_path), 'metrics_path': str(metrics_path), 'feature_importance_path': str(importance_path)})

## 12. Save model manifest

In [ ]:
manifest = build_model_manifest(
    model_name='traffic_forecasting_lightgbm',
    model_type='LightGBMRegressor',
    dataset_path=str(dataset_path),
    target_column=target_column,
    feature_columns=result.feature_selection.feature_columns,
    metrics=result.metrics,
    artifact_path=str(model_path),
)
manifest_path = save_model_manifest(manifest, METADATA_DIR if not KAGGLE_MODE else OUTPUT_DIR, 'model_manifest.json')
display(manifest)
print('Saved manifest:', manifest_path)

## 13. Optional MLflow logging

If `MLFLOW_TRACKING_URI` is set, the notebook logs metrics and artifacts. On Kaggle/Colab without a tracking server, logging is skipped gracefully.

In [ ]:
artifact_paths = [model_path, metrics_path, importance_path, manifest_path]
if feature_importance_png is not None:
    artifact_paths.append(feature_importance_png)

mlflow_status = log_training_run(
    experiment_name=os.getenv('MLFLOW_EXPERIMENT_NAME', 'traffic-notebook-training'),
    run_name='notebook-lightgbm-traffic-forecast',
    dataset_path=str(dataset_path),
    dataset_row_count=len(df),
    feature_columns=result.feature_selection.feature_columns,
    model_type='LightGBMRegressor',
    hyperparameters=result.hyperparameters,
    metrics=result.metrics,
    artifact_paths=artifact_paths,
)
mlflow_status

## 14. Conclusion and limitations

The notebook trains a reusable LightGBM traffic forecasting model and writes portable artifacts. The model is suitable for local portfolio experimentation, but real-world predictive validity depends on data quality, coverage, and whether the training data is synthetic, calibrated, or live operational data. This notebook does not require Docker, Kafka, Neo4j, or MLflow to run on Kaggle.